# DIAG-002 - CPU-only post-hoc T2ICount analysis

Read completed DIAG-001 artifacts: 230 images, 1,380 predictions, 14,490 keyed map
comparisons, and 120 saved NPZs on the original 20 images. No GPU, model/checkpoint
loading, inference, training, prompt generation, or research-ledger edits.

This notebook carries the exact reviewed CPU analysis script as an embedded fallback,
so it can run in VS Code/Colab before the researcher commits/pushes DIAG-002. No source
ZIP or source checkout is required. It prefers an identical local copy when present.
The embedded/source SHA256 is verified; no code is downloaded or guessed.

Class-name inference is primary. Detailed prompts are secondary. Dot localization
metrics are unavailable. Numeric associations do not establish causal mechanisms.


In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess, tempfile, importlib.util, zipfile, shutil, atexit

ASSET_ROOT = Path('/content/drive/MyDrive/T2ICount-assets')
INPUT_DIR = ASSET_ROOT / 'outputs/spatial_diagnostic_v2_all230'
OUTPUT_DIR = INPUT_DIR / 'analysis_cpu_v1'
# For a local kernel, set INPUT_DIR to an existing downloaded artifact directory.
IMAGE_ROOT = None  # Existing matching original RGB directory; auto-discover below.
ASSET_ZIP = Path('/content/drive/MyDrive/T2ICount-assets.zip')
LOCAL_ANALYSIS_SCRIPT = Path.cwd() / 'tools/analyze_spatial_diagnostic.py'
RUN_ANALYSIS = True  # CPU only; existing completed results are reused, never overwritten.
REUSE_COMPLETED = True
VALIDATE_ONLY = False
INSTALL_MISSING_CPU_PACKAGES = True
BOOTSTRAP_REPETITIONS = 2000
SEED = 3407

def stream(command):
    process=subprocess.Popen(list(map(str,command)),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                             text=True,bufsize=1)
    for line in process.stdout: print(line,end='',flush=True)
    if process.wait(): raise subprocess.CalledProcessError(process.returncode,command)


## Mount/check existing artifacts and CPU dependencies
No CUDA availability check or Torch import is needed. Only NumPy, pandas, SciPy,
Matplotlib and Pillow are used. Existing model/checkpoint packages are irrelevant.


In [ ]:
if not INPUT_DIR.is_dir() and str(INPUT_DIR).startswith('/content/drive/'):
    from google.colab import drive
    drive.mount('/content/drive')
assert INPUT_DIR.is_dir(), f'Existing DIAG-001 folder missing: {INPUT_DIR}'
packages={'numpy':'numpy>=1.23,<3','pandas':'pandas>=1.5,<3','scipy':'scipy>=1.9,<2',
          'matplotlib':'matplotlib>=3.5,<4','PIL':'pillow>=9,<12'}
missing=[requirement for module,requirement in packages.items() if importlib.util.find_spec(module) is None]
if missing:
    assert INSTALL_MISSING_CPU_PACKAGES, f'Missing CPU dependencies: {missing}'
    stream([sys.executable,'-m','pip','install',*missing])
print('Input artifacts:', INPUT_DIR)
print('Analysis destination:', OUTPUT_DIR)
print('This pipeline does not import a model or run dataset inference.')


## Load the reviewed standalone CPU implementation
The fallback is generated from `tools/analyze_spatial_diagnostic.py`. Temporary source
and optional RGB extraction folders are private local scratch and cleaned on kernel exit.
Nothing is written into the original maps, figures, CSVs, or JSON inputs.


In [ ]:
EXPECTED_ANALYSIS_SHA256 = '93c4e7a419c15501694bb47c677e907c4e2dac705d619ec42b7d868dfe92177c'
EMBEDDED_ANALYSIS_SOURCE = '#!/usr/bin/env python\n"""DIAG-002: CPU-only post-hoc analysis of immutable DIAG-001 artifacts.\n\nThis module deliberately imports no T2ICount/model/checkpoint/torch utilities.\nIt also runs standalone when embedded in the analysis notebook.\n"""\nimport argparse\nimport csv\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport itertools\nimport json\nimport logging\nimport math\nfrom pathlib import Path\nimport subprocess\nimport sys\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy.stats import rankdata\n\nMODELS = (\'BASE-002\', \'RP-003\', \'RP-009\')\nPROMPTS = (\'class\', \'detailed\')\nMAP_TYPES = (\'fused_attention\', \'sim_x2\', \'sim_x1\', \'density_native\',\n             \'attention_12\', \'attention_24\', \'attention_48\')\nORIGINAL_IMAGES = (\'3319.jpg\', \'7706.jpg\', \'918.jpg\', \'31.jpg\', \'605.jpg\',\n                   \'290.jpg\', \'343.jpg\', \'336.jpg\', \'6305.jpg\', \'3169.jpg\',\n                   \'6353.jpg\', \'3721.jpg\', \'3699.jpg\', \'3019.jpg\', \'5651.jpg\',\n                   \'63.jpg\', \'3287.jpg\', \'75.jpg\', \'3703.jpg\', \'5356.jpg\')\nBINS = (\'GT=1\', \'GT=2-5\', \'GT=6-20\', \'GT>20\')\nEPSILON = 1e-12  # Minimum positive total density mass; never added to/clipped into D.\nTIE_TOLERANCE = 1e-9\nSCHEMA = \'diag002-cpu-v1\'\n\n\nclass ValidationError(ValueError):\n    def __init__(self, issues):\n        self.issues = list(issues)\n        super().__init__(\'Input validation failed:\\n- \' + \'\\n- \'.join(self.issues))\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef json_write(path, value):\n    def clean(item):\n        if isinstance(item,dict): return {str(k):clean(v) for k,v in item.items()}\n        if isinstance(item,(list,tuple)): return [clean(v) for v in item]\n        if isinstance(item,np.generic): return clean(item.item())\n        if isinstance(item,float) and not math.isfinite(item): return None\n        return item\n    Path(path).write_text(json.dumps(clean(value), indent=2, ensure_ascii=False,\n                                    allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n\n\ndef csv_write(path, rows):\n    if isinstance(rows, pd.DataFrame):\n        rows = rows.to_dict(\'records\')\n    if not rows:\n        raise ValueError(\'Refusing an empty table without a declared schema: \' + str(path))\n    with Path(path).open(\'w\', newline=\'\', encoding=\'utf-8\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n        writer.writeheader()\n        for row in rows:\n            writer.writerow({k: \'\' if v is None or (isinstance(v, float) and not math.isfinite(v)) else v\n                             for k, v in row.items()})\n\n\ndef bin_name(count):\n    if count == 1:\n        return BINS[0]\n    if 2 <= count <= 5:\n        return BINS[1]\n    if 6 <= count <= 20:\n        return BINS[2]\n    if count > 20:\n        return BINS[3]\n    raise ValueError(\'GT count outside the approved bins: \' + str(count))\n\n\ndef condition_pairs():\n    conditions = list(itertools.product(MODELS, PROMPTS))\n    return [(left, right) for i, left in enumerate(conditions) for right in conditions[i+1:]\n            if left[0] == right[0] or left[1] == right[1]]\n\n\ndef archive_name(image, model, prompt):\n    if Path(image).name != image or model not in MODELS or prompt not in PROMPTS:\n        raise ValueError(\'Unsafe artifact key\')\n    return \'{}__{}__{}.npz\'.format(Path(image).stem, model, prompt)\n\n\ndef expected_coordinates(height, width):\n    if min(height, width) < 384:\n        raise ValueError(\'Image smaller than original inference patch\')\n    return [[min(j*384, width-384), min(i*384, height-384), 384, 384]\n            for i in range((height-1)//384+1) for j in range((width-1)//384+1)]\n\n\ndef normalize_density(density, epsilon=EPSILON):\n    """No clipping or pseudo target: invalid probability assumptions stay explicit."""\n    density = np.asarray(density, dtype=np.float64)\n    if not np.isfinite(density).all():\n        return None, \'nonfinite\'\n    if np.any(density < 0):\n        return None, \'negative_values\'\n    mass = float(density.sum())\n    if mass <= epsilon:\n        return None, \'zero_or_near_zero_mass\'\n    return density / mass, \'valid\'\n\n\ndef correlation(x, y, method=\'pearson\', controls=None):\n    """Pairwise complete cases; partial Spearman = OLS residuals of ranks.\n\n    Controls are log1p(GT), log1p(reference absolute error). Ranks use average\n    ties. No p-values/multiple-comparison selection or predictive fitting.\n    """\n    x, y = np.asarray(x, float), np.asarray(y, float)\n    valid = np.isfinite(x) & np.isfinite(y)\n    if controls is not None:\n        controls = np.asarray(controls, float)\n        valid &= np.isfinite(controls).all(1)\n    x, y = x[valid], y[valid]\n    n, rank = len(x), 0\n    if n < 3:\n        return {\'r\': None, \'n\': n, \'status\': \'insufficient_n\', \'control_rank\': rank}\n    if method == \'spearman\':\n        x, y = rankdata(x), rankdata(y)\n    elif method != \'pearson\':\n        raise ValueError(\'Unknown correlation method\')\n    if controls is not None:\n        covariates = controls[valid]\n        if method == \'spearman\':\n            covariates = np.column_stack([rankdata(covariates[:, j]) for j in range(covariates.shape[1])])\n        design = np.column_stack([np.ones(n), covariates])\n        rank = int(np.linalg.matrix_rank(design))\n        if n <= rank + 2:\n            return {\'r\': None, \'n\': n, \'status\': \'insufficient_residual_df\', \'control_rank\': rank}\n        x = x - design @ np.linalg.lstsq(design, x, rcond=None)[0]\n        y = y - design @ np.linalg.lstsq(design, y, rcond=None)[0]\n    # Relative tolerance handles exact linear predictions after residualization.\n    if np.linalg.norm(x-x.mean()) <= 1e-10 * max(1., np.linalg.norm(x)) or \\\n            np.linalg.norm(y-y.mean()) <= 1e-10 * max(1., np.linalg.norm(y)):\n        return {\'r\': None, \'n\': n, \'status\': \'constant_or_zero_residual\', \'control_rank\': rank}\n    return {\'r\': float(np.corrcoef(x, y)[0, 1]), \'n\': n, \'status\': \'valid\', \'control_rank\': rank}\n\n\ndef bootstrap_mean(values, repetitions=2000, seed=3407):\n    values = np.asarray(values, dtype=float)\n    if len(values) == 0 or repetitions < 1:\n        raise ValueError(\'Bootstrap requires observations and positive repetitions\')\n    rng = np.random.default_rng(seed)\n    means = values[rng.integers(0, len(values), size=(repetitions, len(values)))].mean(1)\n    return float(np.quantile(means, .025)), float(np.quantile(means, .975))\n\n\ndef _read_table(path, columns, numeric, nullable=()):\n    try:\n        table = pd.read_csv(path, keep_default_na=False)\n    except (OSError, ValueError) as exc:\n        raise ValidationError([str(path) + \': \' + str(exc)]) from exc\n    missing = set(columns) - set(table.columns)\n    if missing:\n        raise ValidationError([path.name + \': missing columns \' + repr(sorted(missing))])\n    for key in numeric:\n        empty = table[key].eq(\'\')\n        try:\n            table[key] = pd.to_numeric(table[key].replace(\'\', np.nan), errors=\'raise\')\n        except (ValueError, TypeError) as exc:\n            raise ValidationError([path.name + \': nonnumeric \' + key]) from exc\n        allowed_missing = empty if key in nullable else np.zeros(len(table), dtype=bool)\n        if ((~np.isfinite(table[key])) & ~allowed_missing).any():\n            raise ValidationError([path.name + \': non-finite \' + key])\n    return table\n\n\ndef validate_tables(root):\n    """Require exact original keys/coverage; never fix rows, prompts or source summaries."""\n    root = Path(root)\n    try:\n        manifest = json.loads((root/\'manifest.json\').read_text(encoding=\'utf-8\'))\n        summary = json.loads((root/\'summary.json\').read_text(encoding=\'utf-8\'))\n    except (ValueError, OSError) as exc:\n        raise ValidationError([\'Missing/unreadable manifest or summary: \' + str(exc)]) from exc\n    issues = []\n    if manifest.get(\'status\') != \'complete\' or summary.get(\'status\') != \'complete\':\n        issues.append(\'DIAG-001 is not complete\')\n    if manifest.get(\'schema_version\') != \'fsc147s-spatial-v2\':\n        issues.append(\'Require actual full-230 fsc147s-spatial-v2 schema\')\n    records = manifest.get(\'selected_images\', [])\n    ids = [r.get(\'image\') for r in records]\n    if len(ids) != 230 or len(set(ids)) != 230 or any(not isinstance(i, str) or Path(i).name != i for i in ids):\n        issues.append(\'Require 230 unique safe image identifiers\')\n    if manifest.get(\'visualization\', {}).get(\'selected_images\') != list(ORIGINAL_IMAGES) or \\\n            manifest.get(\'visualization\', {}).get(\'count\') != 20:\n        issues.append(\'Original 20 visualization images changed/missing\')\n    if manifest.get(\'config\', {}).get(\'models\') != list(MODELS) or manifest.get(\'config\', {}).get(\'prompts\') != list(PROMPTS):\n        issues.append(\'Exact three checkpoint/two prompt identifiers required\')\n    inference = manifest.get(\'inference\', {})\n    if inference.get(\'patch_size\') != 384 or inference.get(\'stride\') != 384:\n        issues.append(\'Incompatible patch geometry\')\n    if issues:\n        raise ValidationError(issues)\n    record_by_id = {r[\'image\']: r for r in records}\n    for record in records:\n        try:\n            gt = float(record[\'gt_count\'])\n            if not math.isfinite(gt) or int(gt) != gt:\n                raise ValueError(\'non-integer/non-finite GT\')\n            bin_name(gt)\n            h, w = record[\'image_hw\']\n            if record[\'patch_xywh\'] != expected_coordinates(h, w):\n                raise ValueError(\'patch coordinates disagree with production extraction\')\n            for prompt in PROMPTS:\n                token = record[\'prompts\'][prompt]\n                active = token[\'active_token_indices\']\n                if (len(token[\'input_ids\']) != 77 or len(token[\'tokens\']) != 77 or len(token[\'mask\']) != 77\n                        or active != list(range(1, len(active)+1)) or not 1 <= len(active) <= 75\n                        or [i for i,v in enumerate(token[\'mask\']) if v != 0] != active\n                        or not all(token[\'mask\'][i] == 1 for i in active)):\n                    raise ValueError(\'incompatible saved token/mask alignment\')\n        except (KeyError, TypeError, ValueError) as exc:\n            issues.append(record[\'image\'] + \': \' + str(exc))\n    results = _read_table(root/\'results.csv\', (\'image\',\'class\',\'model\',\'prompt_type\',\'prompt\',\'gt_count\',\'pred_count\',\'abs_error\',\'patch_count\'),\n                          (\'gt_count\',\'pred_count\',\'abs_error\',\'patch_count\'))\n    keys = [\'image\',\'model\',\'prompt_type\']\n    actual = set(results[keys].itertuples(index=False, name=None))\n    expected = set(itertools.product(ids, MODELS, PROMPTS))\n    if len(results) != 1380 or results.duplicated(keys).any() or actual != expected:\n        issues.append(\'Counting joins: expected 1380 exact unique image/model/prompt keys; missing={}, extra={}, duplicates={}\'.format(\n            len(expected-actual), len(actual-expected), int(results.duplicated(keys).sum())))\n    if actual == expected:\n        for row in results.itertuples():\n            record = record_by_id[row.image]\n            if row.gt_count != record[\'gt_count\']:\n                issues.append(row.image + \': GT mismatch\')\n            if row.prompt != record[\'prompts\'][row.prompt_type][\'prompt\']:\n                issues.append(row.image + \': prompt text mismatch\')\n            if row.patch_count != len(record[\'patch_xywh\']):\n                issues.append(row.image + \': patch count mismatch\')\n        if any(row[\'class\'] != record_by_id[row[\'image\']][\'class\'] for row in results.to_dict(\'records\')):\n            issues.append(\'Class mapping mismatch\')\n        if not np.allclose(results.abs_error, np.abs(results.pred_count-results.gt_count), rtol=1e-9, atol=1e-9):\n            issues.append(\'Stored absolute errors disagree with predictions/GT\')\n    maps = _read_table(root/\'map_comparisons.csv\', (\'image\',\'left_model\',\'left_prompt\',\'right_model\',\'right_prompt\',\'map_type\',\'normalized_l1\',\'pearson\'),\n                       (\'normalized_l1\',\'pearson\'), nullable=(\'pearson\',))\n    map_keys = [\'image\',\'left_model\',\'left_prompt\',\'right_model\',\'right_prompt\',\'map_type\']\n    expected_maps = {(image,*left,*right,kind) for image in ids for left,right in condition_pairs() for kind in MAP_TYPES}\n    actual_maps = set(maps[map_keys].itertuples(index=False, name=None))\n    if len(maps) != 14490 or maps.duplicated(map_keys).any() or actual_maps != expected_maps:\n        issues.append(\'Map joins: expected 14490 exact unique keys; missing={}, extra={}, duplicates={}\'.format(\n            len(expected_maps-actual_maps),len(actual_maps-expected_maps),int(maps.duplicated(map_keys).sum())))\n    if (maps.normalized_l1.lt(0) | maps.normalized_l1.gt(1+1e-7)).any() or \\\n            (maps.pearson.dropna().abs() > 1+1e-7).any():\n        issues.append(\'Normalized map metric outside its documented range\')\n    summary_counts = {(r[\'model\'],r[\'prompt_type\']):r for r in summary.get(\'counting\',[])}\n    if len(summary.get(\'counting\',[])) != 6 or set(summary_counts) != set(itertools.product(MODELS,PROMPTS)):\n        issues.append(\'Counting summary condition coverage mismatch\')\n    else:\n        for condition, data in results.groupby([\'model\',\'prompt_type\']):\n            item = summary_counts[condition]\n            if item.get(\'n\') != 230 or not np.allclose([item.get(\'mae\'),item.get(\'rmse\')],\n                    [data.abs_error.mean(),np.sqrt(np.mean(data.abs_error**2))], rtol=1e-8,atol=1e-8):\n                issues.append(\'Source summary count metrics disagree: \' + str(condition))\n    aggregate_keys=[\'left_model\',\'left_prompt\',\'right_model\',\'right_prompt\',\'map_type\']\n    saved_aggregates=summary.get(\'map_comparisons\',[])\n    saved_lookup={tuple(row[k] for k in aggregate_keys):row for row in saved_aggregates}\n    if len(saved_aggregates)!=63 or set(saved_lookup)!= {(*left,*right,kind) for left,right in condition_pairs() for kind in MAP_TYPES}:\n        issues.append(\'Source map summary coverage mismatch\')\n    else:\n        for key,data in maps.groupby(aggregate_keys,sort=True):\n            item=saved_lookup[key];valid=data.pearson.dropna()\n            expected_pearson=float(valid.mean()) if len(valid) else None\n            if (item.get(\'n\')!=230 or item.get(\'pearson_valid_n\')!=len(valid)\n                    or not np.isclose(item.get(\'mean_normalized_l1\',np.nan),data.normalized_l1.mean(),rtol=1e-8,atol=1e-8)\n                    or (expected_pearson is None and item.get(\'mean_pearson\') is not None)\n                    or (expected_pearson is not None and not np.isclose(item.get(\'mean_pearson\',np.nan),expected_pearson,rtol=1e-8,atol=1e-8))):\n                issues.append(\'Source map summary disagrees with keyed records: \'+str(key))\n    expected_npz = {archive_name(i,m,p) for i in ORIGINAL_IMAGES for m,p in itertools.product(MODELS,PROMPTS)}\n    if {p.name for p in (root/\'maps\').glob(\'*.npz\')} != expected_npz:\n        issues.append(\'Require exactly 120 NPZ files for the unchanged 20 images\')\n    capture = manifest.get(\'capture_equivalence\',[])\n    if len(capture)!=6 or {(r[\'model\'],r[\'image\'],r[\'prompt\']) for r in capture} != \\\n            {(m,\'3319.jpg\',p) for m,p in itertools.product(MODELS,PROMPTS)} or not all(r[\'bitwise_equal\'] is True for r in capture):\n        issues.append(\'DIAG-001 designated-image capture checks incomplete\')\n    if issues:\n        raise ValidationError(issues)\n    results[\'gt_bin\'] = results.gt_count.map(bin_name)\n    # Explicit stable joins; CSV row order never determines pairing or bootstrap.\n    results = results.sort_values(keys).reset_index(drop=True)\n    maps = maps.sort_values(map_keys).reset_index(drop=True)\n    return manifest, summary, results, maps\n\n\ndef validate_npz(root, manifest, results, image_root=None):\n    records = {r[\'image\']:r for r in manifest[\'selected_images\']}\n    lookup = results.set_index([\'image\',\'model\',\'prompt_type\'])\n    issues, statuses, images = [], [], {}\n    for image in ORIGINAL_IMAGES:\n        record = records[image]\n        image_path = Path(image_root)/image if image_root else Path(record[\'image_file\'][\'path\'])\n        if image_path.is_file():\n            if sha256(image_path) != record[\'image_file\'][\'sha256\']:\n                issues.append(image + \': RGB source fingerprint mismatch\')\n            else:\n                with Image.open(image_path) as im:\n                    if list(im.size) != record[\'image_hw\'][::-1]:\n                        issues.append(image + \': RGB dimension mismatch\')\n                images[image] = image_path.resolve()\n        else:\n            issues.append(image + \': original RGB unavailable; set --image-root to existing matching images\')\n        for model,prompt in itertools.product(MODELS,PROMPTS):\n            path = Path(root)/\'maps\'/archive_name(image,model,prompt)\n            try:\n                with np.load(path,allow_pickle=False) as data:\n                    expected = set(MAP_TYPES)|{\'tokens_12\',\'tokens_24\',\'tokens_48\',\'density_full\',\'patch_xywh\'}\n                    if set(data.files) != expected:\n                        raise ValueError(\'NPZ array keys mismatch: \' + str(set(data.files)^expected))\n                    patches, tokens = len(record[\'patch_xywh\']), len(record[\'prompts\'][prompt][\'active_token_indices\'])\n                    for kind in expected:\n                        value = data[kind]\n                        if value.dtype.kind not in \'fiu\' or not np.isfinite(value).all():\n                            raise ValueError(\'nonnumeric/nonfinite \' + kind)\n                        if kind==\'patch_xywh\' and value.dtype.kind not in \'iu\':\n                            raise ValueError(\'patch coordinates must be integer arrays\')\n                        if kind.startswith(\'tokens_\'):\n                            size = int(kind.split(\'_\')[-1]); shape=(patches,tokens,size,size)\n                        elif kind.startswith(\'attention_\'):\n                            size = int(kind.split(\'_\')[-1]); shape=(patches,1,size,size)\n                        elif kind==\'density_full\': shape=(1,1,*record[\'image_hw\'])\n                        elif kind==\'patch_xywh\': shape=(patches,4)\n                        else: shape=(patches,1,48,48)\n                        if value.shape != shape:\n                            raise ValueError(\'{} shape {} != {}\'.format(kind,value.shape,shape))\n                    if not np.array_equal(data[\'patch_xywh\'],record[\'patch_xywh\']):\n                        raise ValueError(\'patch coordinate mismatch\')\n                    for size in (12,24,48):\n                        if not np.allclose(data[\'attention_\'+str(size)],data[\'tokens_\'+str(size)].mean(1,keepdims=True),rtol=2e-6,atol=1e-7):\n                            raise ValueError(\'content-token mean mismatch at \'+str(size))\n                    mass = float(data[\'density_full\'].sum(dtype=np.float64))\n                    if not np.isclose(mass,lookup.loc[(image,model,prompt),\'pred_count\'],rtol=2e-6,atol=1e-5):\n                        raise ValueError(\'full density mass disagrees with source counting/scaling\')\n                    probability,status = normalize_density(data[\'density_full\'])\n                    statuses.append({\'image\':image,\'model\':model,\'prompt_type\':prompt,\'mass\':mass,\n                                     \'min_density\':float(data[\'density_full\'].min()),\'normalization_status\':status})\n            except (OSError,ValueError,KeyError) as exc:\n                issues.append(path.name+\': \'+str(exc))\n    if issues:\n        raise ValidationError(issues)\n    return statuses, images\n\n\ndef counting_tables(results, repetitions=2000):\n    summaries, paired, pair_summaries, prompt_rows, winner_rows = [],[],[],[],[]\n    for (model,prompt), group in results.groupby([\'model\',\'prompt_type\'],sort=True):\n        for name, data in [(\'all\',group)]+[(b,group[group.gt_bin==b]) for b in BINS]:\n            signed = (data.pred_count-data.gt_count).to_numpy()\n            n=len(data)\n            summaries.append({\'model\':model,\'prompt_type\':prompt,\'gt_bin\':name,\'n\':n,\n                \'mae\':float(data.abs_error.mean()) if n else None,\n                \'rmse\':float(np.sqrt(np.mean(data.abs_error**2))) if n else None,\n                \'signed_bias\':float(signed.mean()) if n else None,\n                \'median_absolute_error\':float(data.abs_error.median()) if n else None,\n                \'over_n\':int(np.sum(signed>TIE_TOLERANCE)),\'under_n\':int(np.sum(signed<-TIE_TOLERANCE)),\n                \'exact_n\':int(np.sum(np.abs(signed)<=TIE_TOLERANCE)),\n                \'over_pct\':float(100*np.mean(signed>TIE_TOLERANCE)) if n else None,\n                \'under_pct\':float(100*np.mean(signed<-TIE_TOLERANCE)) if n else None,\n                \'over_error_mean\':float(np.maximum(signed,0).mean()) if n else None,\n                \'under_error_mean\':float(np.maximum(-signed,0).mean()) if n else None,\n                \'pred_mean\':float(data.pred_count.mean()) if n else None,\n                **{\'pred_\'+key:float(data.pred_count.quantile(q)) if n else None\n                   for key,q in [(\'p05\',.05),(\'p25\',.25),(\'median\',.5),(\'p75\',.75),(\'p95\',.95)]},\n                \'pred_max\':float(data.pred_count.max()) if n else None})\n    lookup=results.set_index([\'image\',\'model\',\'prompt_type\'])\n    ids=sorted(results.image.unique())\n    for prompt in PROMPTS:\n        for reference,candidate in itertools.combinations(MODELS,2):\n            these=[]\n            for image in ids:\n                left,right=lookup.loc[(image,reference,prompt)],lookup.loc[(image,candidate,prompt)]\n                gain=float(left.abs_error-right.abs_error)\n                row={\'image\':image,\'prompt_type\':prompt,\'reference\':reference,\'candidate\':candidate,\n                     \'gt_count\':float(left.gt_count),\'gt_bin\':left.gt_bin,\n                     \'pred_reference\':float(left.pred_count),\'pred_candidate\':float(right.pred_count),\n                     \'error_reference\':float(left.abs_error),\'error_candidate\':float(right.abs_error),\n                     \'delta_pred\':float(right.pred_count-left.pred_count),\'error_gain\':gain,\n                     \'outcome\':\'better\' if gain>TIE_TOLERANCE else \'worse\' if gain<-TIE_TOLERANCE else \'tie\',\n                     \'over_error_gain\':float(max(left.pred_count-left.gt_count,0)-max(right.pred_count-right.gt_count,0)),\n                     \'under_error_gain\':float(max(left.gt_count-left.pred_count,0)-max(right.gt_count-right.pred_count,0))}\n                paired.append(row);these.append(row)\n            for name,data in [(\'all\',these)]+[(b,[r for r in these if r[\'gt_bin\']==b]) for b in BINS]:\n                n=len(data)\n                gains=np.array([r[\'error_gain\'] for r in data])\n                lo,hi=bootstrap_mean(gains,repetitions) if name==\'all\' else (None,None)\n                pair_summaries.append({\'reference\':reference,\'candidate\':candidate,\'prompt_type\':prompt,\'gt_bin\':name,\'n\':n,\n                    \'candidate_wins\':sum(r[\'outcome\']==\'better\' for r in data),\n                    \'candidate_loses\':sum(r[\'outcome\']==\'worse\' for r in data),\'ties\':sum(r[\'outcome\']==\'tie\' for r in data),\n                    \'mean_error_gain\':float(gains.mean()) if n else None,\'gain_ci_low\':lo,\'gain_ci_high\':hi,\n                    \'mean_delta_pred\':float(np.mean([r[\'delta_pred\'] for r in data])) if n else None,\n                    \'prediction_lower_n\':sum(r[\'delta_pred\']<-TIE_TOLERANCE for r in data),\n                    \'mean_over_error_gain\':float(np.mean([r[\'over_error_gain\'] for r in data])) if n else None,\n                    \'mean_under_error_gain\':float(np.mean([r[\'under_error_gain\'] for r in data])) if n else None})\n        for model in MODELS:\n            sole,shared,not_best=0,0,0\n            for image in ids:\n                errors=np.array([lookup.loc[(image,m,prompt),\'abs_error\'] for m in MODELS])\n                best=np.abs(errors-errors.min())<=TIE_TOLERANCE\n                if best[MODELS.index(model)]:\n                    if best.sum()==1: sole+=1\n                    else: shared+=1\n                else: not_best+=1\n            winner_rows.append({\'model\':model,\'prompt_type\':prompt,\'n\':len(ids),\n                                \'sole_best\':sole,\'shared_best\':shared,\'not_best\':not_best})\n    for model in MODELS:\n        for image in ids:\n            left,right=lookup.loc[(image,model,\'class\')],lookup.loc[(image,model,\'detailed\')]\n            prompt_rows.append({\'image\':image,\'model\':model,\'gt_count\':float(left.gt_count),\'gt_bin\':left.gt_bin,\n                \'class_pred\':float(left.pred_count),\'detailed_pred\':float(right.pred_count),\n                \'class_error\':float(left.abs_error),\'detailed_error\':float(right.abs_error),\n                \'delta_pred\':float(right.pred_count-left.pred_count),\'delta_abs_error\':float(right.abs_error-left.abs_error)})\n    return {\'counting_summary\':pd.DataFrame(summaries),\'paired_counting\':pd.DataFrame(paired),\n            \'paired_counting_summary\':pd.DataFrame(pair_summaries),\'prompt_sensitivity\':pd.DataFrame(prompt_rows),\n            \'model_winners\':pd.DataFrame(winner_rows)}\n\n\ndef spatial_tables(results,maps):\n    lookup=results.set_index([\'image\',\'model\',\'prompt_type\'])\n    joined=[]\n    for item in maps.to_dict(\'records\'):\n        left=lookup.loc[(item[\'image\'],item[\'left_model\'],item[\'left_prompt\'])]\n        right=lookup.loc[(item[\'image\'],item[\'right_model\'],item[\'right_prompt\'])]\n        within=item[\'left_model\']==item[\'right_model\']\n        joined.append({**item,\'comparison\':item[\'left_model\']+\'/\'+item[\'left_prompt\']+\' -> \'+item[\'right_model\']+\'/\'+item[\'right_prompt\'],\n                       \'kind\':\'prompt\' if within else \'checkpoint\',\n                       \'gt_count\':float(left.gt_count),\'gt_bin\':left.gt_bin,\n                       \'reference_error\':float(left.abs_error),\'error_gain\':float(left.abs_error-right.abs_error),\n                       \'delta_abs_error\':float(right.abs_error-left.abs_error),\n                       \'delta_pred\':float(right.pred_count-left.pred_count),\n                       \'one_minus_pearson\':float(1-item[\'pearson\']) if np.isfinite(item[\'pearson\']) else np.nan})\n    joined=pd.DataFrame(joined)\n    associations, groups, extremes=[],[],[]\n    for (comparison,kind,map_type),data in joined.groupby([\'comparison\',\'kind\',\'map_type\'],sort=True):\n        outcomes=np.where(data.error_gain>TIE_TOLERANCE,\'better\',np.where(data.error_gain<-TIE_TOLERANCE,\'worse\',\'tie\'))\n        for label,selected in [(\'all\',data)]+[(name,data[outcomes==name]) for name in (\'better\',\'worse\',\'tie\')]:\n            valid=selected.pearson.dropna()\n            groups.append({\'comparison\':comparison,\'kind\':kind,\'map_type\':map_type,\'candidate_outcome\':label,\n                \'n\':len(selected),\'pearson_valid_n\':len(valid),\n                \'pearson_mean\':float(valid.mean()) if len(valid) else None,\n                \'pearson_median\':float(valid.median()) if len(valid) else None,\n                **{\'l1_\'+key:float(selected.normalized_l1.quantile(q)) if len(selected) else None\n                   for key,q in [(\'p25\',.25),(\'median\',.5),(\'p75\',.75)]}})\n        targets=[\'delta_pred\',\'delta_abs_error\'] if kind==\'prompt\' else [\'error_gain\']\n        for target in targets:\n            for feature in (\'normalized_l1\',\'one_minus_pearson\'):\n                strata=[(\'all\',data)]+[(b,data[data.gt_bin==b]) for b in BINS]\n                for stratum,selected in strata:\n                    for method in (\'pearson\',\'spearman\'):\n                        for adjusted in ([False,True] if stratum==\'all\' else [False]):\n                            controls=np.column_stack([np.log1p(selected.gt_count),np.log1p(selected.reference_error)]) if adjusted else None\n                            result=correlation(selected[feature],selected[target],method,controls)\n                            associations.append({\'comparison\':comparison,\'kind\':kind,\'map_type\':map_type,\n                                \'feature\':feature,\'target\':target,\'gt_bin\':stratum,\'method\':method,\n                                \'adjustment\':\'log1p_gt_and_reference_error\' if adjusted else \'none\',**result})\n        # Fixed top three per direction/metric, identities retained for inspection.\n        for metric,ascending in [(\'normalized_l1\',False),(\'pearson\',True)]:\n            for row in data.dropna(subset=[metric]).sort_values([metric,\'image\'],ascending=[ascending,True]).head(3).to_dict(\'records\'):\n                extremes.append({\'comparison\':comparison,\'map_type\':map_type,\'selection_metric\':metric,\n                                 \'image\':row[\'image\'],\'normalized_l1\':float(row[\'normalized_l1\']),\n                                 \'pearson\':float(row[\'pearson\']) if np.isfinite(row[\'pearson\']) else None,\n                                 \'error_gain\':float(row[\'error_gain\']),\'gt_count\':float(row[\'gt_count\'])})\n    return {\'spatial_joined\':joined,\'spatial_associations\':pd.DataFrame(associations),\n            \'spatial_groups\':pd.DataFrame(groups),\'spatial_extremes\':pd.DataFrame(extremes)}\n\n\ndef density_shapes(root,statuses):\n    rows=[]\n    for image in ORIGINAL_IMAGES:\n        densities={}\n        for condition in itertools.product(MODELS,PROMPTS):\n            with np.load(Path(root)/\'maps\'/archive_name(image,*condition),allow_pickle=False) as data:\n                densities[condition]=data[\'density_full\'][0,0].astype(np.float64)\n        for left,right in condition_pairs():\n            first,second=densities[left],densities[right]\n            p,pstatus=normalize_density(first);q,qstatus=normalize_density(second)\n            mass1,mass2=float(first.sum()),float(second.sum())\n            valid=p is not None and q is not None\n            rows.append({\'image\':image,\'left_model\':left[0],\'left_prompt\':left[1],\n                \'right_model\':right[0],\'right_prompt\':right[1],\n                \'mass_reference\':mass1,\'mass_candidate\':mass2,\'delta_mass\':mass2-mass1,\n                \'mass_ratio\':mass2/mass1 if mass1>EPSILON else None,\n                \'reference_normalization\':pstatus,\'candidate_normalization\':qstatus,\n                \'shape_valid\':valid,\'total_variation\':float(.5*np.abs(p-q).sum()) if valid else None,\n                \'shape_pearson\':correlation(p.ravel(),q.ravel())[\'r\'] if valid else None})\n    return pd.DataFrame(rows)\n\n\ndef save_figure(fig,path,qa):\n    """Render every figure; detect clipped labels and blank/corrupt PNGs."""\n    import matplotlib.pyplot as plt\n    from matplotlib.text import Text\n    fig.canvas.draw()\n    renderer=fig.canvas.get_renderer()\n    width,height=fig.canvas.get_width_height()\n    outside=[]\n    # Axis objects cache tick labels beyond the view limits; those are not\n    # rendered by Matplotlib and must not be mistaken for clipped visible text.\n    hidden_ticks=set()\n    for axis_container in fig.axes:\n        for axis in (axis_container.xaxis,axis_container.yaxis):\n            low,high=sorted(axis.get_view_interval())\n            for tick in axis.get_major_ticks()+axis.get_minor_ticks():\n                if not axis_container.axison or tick.get_loc()<low or tick.get_loc()>high:\n                    hidden_ticks.update((id(tick.label1),id(tick.label2)))\n    for text in fig.findobj(Text):\n        if id(text) in hidden_ticks or not text.get_visible() or not text.get_text().strip():\n            continue\n        box=text.get_window_extent(renderer)\n        if np.isfinite([box.x0,box.y0,box.x1,box.y1]).all() and \\\n                (box.x0 < -3 or box.y0 < -3 or box.x1 > width+3 or box.y1 > height+3):\n            outside.append(text.get_text())\n    fig.savefig(path,dpi=120,facecolor=\'white\')\n    plt.close(fig)\n    with Image.open(path) as image:\n        image.load()\n        extrema=image.convert(\'RGB\').getextrema()\n        if all(low==high for low,high in extrema):\n            raise ValueError(\'Blank figure: \'+str(path))\n        size=list(image.size)\n    qa.append({\'file\':str(Path(path).name),\'pixels\':size,\'clipped_text\':outside,\'sha256\':sha256(path)})\n    if outside:\n        raise ValueError(\'Clipped figure text in {}: {}\'.format(Path(path).name,outside[:4]))\n\n\ndef _matrix(associations,kind,target,adjustment):\n    comparisons=list(dict.fromkeys(associations[associations.kind==kind].comparison))\n    if kind==\'checkpoint\':\n        comparisons=[c for c in comparisons if \'/class -> \' in c and c.endswith(\'/class\')]\n    matrix=np.full((len(MAP_TYPES),len(comparisons)),np.nan)\n    counts=np.zeros(matrix.shape,int)\n    for i,kind_name in enumerate(MAP_TYPES):\n        for j,comparison in enumerate(comparisons):\n            row=associations[(associations.comparison==comparison)&(associations.map_type==kind_name)&\n                (associations.feature==\'normalized_l1\')&(associations.target==target)&\n                (associations.gt_bin==\'all\')&(associations.method==\'spearman\')&\n                (associations.adjustment==adjustment)]\n            if len(row)==1:\n                value=row.iloc[0]\n                matrix[i,j]=value.r if pd.notna(value.r) else np.nan\n                counts[i,j]=value.n\n    return comparisons,matrix,counts\n\n\ndef aggregate_plots(output,results,tables,qa):\n    import matplotlib.pyplot as plt\n    colors=(\'#386cb0\',\'#ef8a62\',\'#4daf4a\')\n    bins_colors=(\'#377eb8\',\'#4daf4a\',\'#ff7f00\',\'#984ea3\')\n    destination=output/\'figures\'\n    fig,axes=plt.subplots(2,3,figsize=(13,8),constrained_layout=True)\n    limit=max(results.pred_count.max(),results.gt_count.max())*1.05\n    for i,prompt in enumerate(PROMPTS):\n        for j,model in enumerate(MODELS):\n            data=results[(results.model==model)&(results.prompt_type==prompt)]\n            ax=axes[i,j]\n            ax.scatter(data.gt_count,data.pred_count,s=13,alpha=.45,color=colors[j])\n            ax.plot([0,limit],[0,limit],\'k--\',lw=.8)\n            ax.set(xscale=\'symlog\',yscale=\'symlog\',xlim=(0,limit),ylim=(min(0,results.pred_count.min()*1.05),limit),\n                   xlabel=\'GT count\',ylabel=\'Predicted count\',title=model+\' / \'+prompt+\' (n=230)\')\n    fig.suptitle(\'Observed counting responses; symlog axes, no dot localization GT\')\n    save_figure(fig,destination/\'count_vs_gt.png\',qa)\n    fig,axes=plt.subplots(2,2,figsize=(12,8),constrained_layout=True)\n    for i,prompt in enumerate(PROMPTS):\n        data=[results[(results.model==m)&(results.prompt_type==prompt)] for m in MODELS]\n        for j,key in enumerate((\'abs_error\',\'signed\')):\n            arrays=[d.abs_error if key==\'abs_error\' else d.pred_count-d.gt_count for d in data]\n            axes[i,j].boxplot(arrays,labels=MODELS,showfliers=True)\n            axes[i,j].set_yscale(\'symlog\',linthresh=1)\n            axes[i,j].set(title=prompt+\' / \'+(\'absolute error\' if j==0 else \'signed error\'),ylabel=\'Count error (symlog)\')\n            axes[i,j].axhline(0,color=\'gray\',lw=.7)\n    fig.suptitle(\'Error distributions; all outliers retained\')\n    save_figure(fig,destination/\'error_distributions.png\',qa)\n    fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)\n    for ax,prompt in zip(axes,PROMPTS):\n        for model,color in zip(MODELS,colors):\n            values=np.sort(results[(results.model==model)&(results.prompt_type==prompt)].pred_count)\n            ax.plot(values,np.arange(1,len(values)+1)/len(values),label=model,color=color)\n        ax.set(xscale=\'symlog\',xlabel=\'Predicted count (symlog)\',ylabel=\'Empirical CDF\',title=prompt)\n        ax.legend(fontsize=8)\n    fig.suptitle(\'Prediction magnitude distributions (n=230 per checkpoint/prompt)\')\n    save_figure(fig,destination/\'prediction_distributions.png\',qa)\n    fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)\n    paired=tables[\'paired_counting\'];class_pairs=paired[paired.prompt_type==\'class\']\n    for (reference,candidate),data in class_pairs.groupby([\'reference\',\'candidate\'],sort=False):\n        for ax,key in zip(axes,(\'error_gain\',\'delta_pred\')):\n            values=np.sort(data[key])\n            ax.plot(values,np.arange(1,len(values)+1)/len(values),label=reference+\' -> \'+candidate)\n            ax.axvline(0,color=\'gray\',lw=.7)\n            ax.set_xscale(\'symlog\',linthresh=1)\n            ax.set_ylabel(\'Empirical CDF\')\n    axes[0].set_xlabel(\'Reference error - candidate error (positive = gain)\')\n    axes[1].set_xlabel(\'Candidate prediction - reference prediction\')\n    for ax in axes: ax.legend(fontsize=8)\n    fig.suptitle(\'Paired class-name counting changes; 230 same-image pairs\')\n    save_figure(fig,destination/\'paired_counting_gains.png\',qa)\n    fig,axes=plt.subplots(1,2,figsize=(12,4),constrained_layout=True)\n    counts=results[(results.model==MODELS[0])&(results.prompt_type==\'class\')].gt_bin.value_counts()\n    for j,model in enumerate(MODELS):\n        data=tables[\'counting_summary\'][(tables[\'counting_summary\'].model==model)&\n             (tables[\'counting_summary\'].prompt_type==\'class\')].set_index(\'gt_bin\')\n        for ax,key in zip(axes,(\'pred_mean\',\'signed_bias\')):\n            ax.bar(np.arange(4)+(j-1)*.25,[data.loc[b,key] for b in BINS],width=.24,label=model,color=colors[j])\n    for ax in axes:\n        ax.set_xticks(range(4));ax.set_xticklabels([b+\'\\nn=\'+str(counts.get(b,0)) for b in BINS])\n        ax.axhline(0,color=\'gray\',lw=.7);ax.legend(fontsize=8)\n    axes[0].set_ylabel(\'Mean predicted count\');axes[1].set_ylabel(\'Mean prediction - GT\')\n    fig.suptitle(\'Class-name prediction magnitude and signed bias by GT bin\')\n    save_figure(fig,destination/\'gt_bin_magnitude_bias.png\',qa)\n    associations=tables[\'spatial_associations\']\n    for kind,target,filename in [(\'checkpoint\',\'error_gain\',\'spatial_association_matrix.png\'),\n                                 (\'prompt\',\'delta_pred\',\'prompt_association_matrix.png\')]:\n        fig,axes=plt.subplots(1,2,figsize=(13,7),constrained_layout=True)\n        for ax,adjustment in zip(axes,(\'none\',\'log1p_gt_and_reference_error\')):\n            comparisons,matrix,valid_n=_matrix(associations,kind,target,adjustment)\n            rendered=ax.imshow(matrix,vmin=-1,vmax=1,cmap=\'coolwarm\',aspect=\'auto\')\n            labels=[c.replace(\'/class\',\'\').replace(\'/detailed\',\'\').replace(\' -> \',\'\\n-> \') for c in comparisons]\n            if kind==\'prompt\': labels=[c.split(\'/\')[0] for c in comparisons]\n            ax.set_xticks(range(len(labels)));ax.set_xticklabels(labels,fontsize=9)\n            ax.set_yticks(range(len(MAP_TYPES)));ax.set_yticklabels(MAP_TYPES,fontsize=9)\n            ax.set_title(\'Raw Spearman\' if adjustment==\'none\' else \'Partial Spearman: GT + reference error\',fontsize=10)\n            for i,j in itertools.product(range(len(MAP_TYPES)),range(len(labels))):\n                label=(\'{:.2f}\'.format(matrix[i,j]) if np.isfinite(matrix[i,j]) else \'NA\')+\'\\nn=\'+str(valid_n[i,j])\n                ax.text(j,i,label,ha=\'center\',va=\'center\',fontsize=9)\n            fig.colorbar(rendered,ax=ax,label=\'Correlation (not causal)\',shrink=.8)\n        fig.suptitle(\'Normalized map L1 association with \'+(\'class counting gain\' if kind==\'checkpoint\' else \'within-model detailed-minus-class count change\'))\n        save_figure(fig,destination/filename,qa)\n    joined=tables[\'spatial_joined\']\n    comparisons=list(dict.fromkeys(joined[(joined.left_prompt==\'class\')&(joined.right_prompt==\'class\')].comparison))\n    fig,axes=plt.subplots(3,2,figsize=(15,11),constrained_layout=True)\n    short=(\'fused\',\'HSCM2\',\'HSCM1\',\'density\',\'attn12\',\'attn24\',\'attn48\')\n    for i,comparison in enumerate(comparisons):\n        data=joined[joined.comparison==comparison]\n        for j,metric in enumerate((\'normalized_l1\',\'pearson\')):\n            ax=axes[i,j]\n            for offset,outcome,color in [(-.18,\'better\',\'#4daf4a\'),(.18,\'worse\',\'#e78ac3\')]:\n                sign=data.error_gain>TIE_TOLERANCE if outcome==\'better\' else data.error_gain<-TIE_TOLERANCE\n                arrays=[data[(data.map_type==kind)&sign][metric].dropna().to_numpy() for kind in MAP_TYPES]\n                arrays=[a if len(a) else np.array([np.nan]) for a in arrays]\n                box=ax.boxplot(arrays,positions=np.arange(7)+offset,widths=.3,patch_artist=True,showfliers=True)\n                for patch in box[\'boxes\']: patch.set_facecolor(color)\n            ax.set_xticks(range(7));ax.set_xticklabels(short,fontsize=8)\n            ax.set_title(comparison.replace(\'/class\',\'\')+\' / \'+metric,fontsize=10)\n    fig.suptitle(\'Class map distributions: green=candidate better, pink=worse; ties in tables, undefined r excluded\')\n    save_figure(fig,destination/\'spatial_group_distributions.png\',qa)\n    # Fixed comparison panels, never choose a pair/map because its correlation is large.\n    for index,comparison in enumerate(comparisons):\n        fig,axes=plt.subplots(2,4,figsize=(15,7),constrained_layout=True)\n        data=joined[joined.comparison==comparison]\n        for ax,kind_name in zip(axes.flat,MAP_TYPES):\n            subset=data[data.map_type==kind_name]\n            for label,color in zip(BINS,bins_colors):\n                part=subset[subset.gt_bin==label]\n                ax.scatter(part.normalized_l1,part.error_gain,s=13,alpha=.55,color=color,label=label+\' n=\'+str(len(part)))\n            ax.axhline(0,color=\'gray\',lw=.6)\n            ax.set(xlabel=\'Normalized map L1\',ylabel=\'Counting error gain\',title=kind_name)\n        axes.flat[-1].axis(\'off\')\n        handles,labels=axes.flat[0].get_legend_handles_labels()\n        axes.flat[-1].legend(handles,labels,loc=\'center\',fontsize=10)\n        fig.suptitle(comparison+\' | class-name association; points are images, not localization labels\')\n        save_figure(fig,destination/\'spatial_scatter_pair{}.png\'.format(index+1),qa)\n    fig,axes=plt.subplots(1,3,figsize=(14,4.5),constrained_layout=True)\n    for ax,model in zip(axes,MODELS):\n        data=tables[\'prompt_sensitivity\'][tables[\'prompt_sensitivity\'].model==model]\n        for label,color in zip(BINS,bins_colors):\n            part=data[data.gt_bin==label]\n            ax.scatter(part.delta_pred,part.delta_abs_error,s=14,alpha=.55,color=color,label=label)\n        ax.axhline(0,color=\'gray\',lw=.6);ax.axvline(0,color=\'gray\',lw=.6)\n        ax.set(xscale=\'symlog\',yscale=\'symlog\',xlabel=\'Detailed - class predicted count\',\n               ylabel=\'Detailed - class absolute error\',title=model)\n        ax.legend(fontsize=8)\n    fig.suptitle(\'Within-checkpoint prompt sensitivity (symlog axes)\')\n    save_figure(fig,destination/\'prompt_count_changes.png\',qa)\n    shape=tables[\'density_shape_pairs\']\n    fig,ax=plt.subplots(figsize=(8,5),constrained_layout=True)\n    for (left,right),data in shape[(shape.left_prompt==\'class\')&(shape.right_prompt==\'class\')].groupby([\'left_model\',\'right_model\']):\n        valid=data[data.shape_valid]\n        ax.scatter(valid.mass_ratio,valid.total_variation,label=left+\' -> \'+right+\' n=\'+str(len(valid)),s=28,alpha=.7)\n    ax.axvline(1,color=\'gray\',lw=.7)\n    ax.set(xlabel=\'Candidate/reference total density mass\',ylabel=\'Total variation of D/sum(D)\',\n           title=\'Original 20 only: magnitude versus normalized shape (no localization GT)\')\n    ax.legend(fontsize=9)\n    save_figure(fig,destination/\'density_mass_vs_shape.png\',qa)\n\n\ndef qualitative_plots(root,output,manifest,results,images,qa):\n    import matplotlib.pyplot as plt\n    from matplotlib.colors import Normalize, PowerNorm\n    records={r[\'image\']:r for r in manifest[\'selected_images\']}\n    lookup=results.set_index([\'image\',\'model\',\'prompt_type\'])\n    raw_max,raw_min=0.,0.\n    for image,model in itertools.product(ORIGINAL_IMAGES,MODELS):\n        with np.load(Path(root)/\'maps\'/archive_name(image,model,\'class\'),allow_pickle=False) as data:\n            raw_max=max(raw_max,float(data[\'density_full\'].max()))\n            raw_min=min(raw_min,float(data[\'density_full\'].min()))\n    raw_max=raw_max or 1.\n    for image in ORIGINAL_IMAGES:\n        logging.getLogger(\'diag002_cpu\').info(\'Qualitative panels: %s\',image)\n        record=records[image]\n        with Image.open(images[image]) as im: rgb=np.asarray(im.convert(\'RGB\'))\n        arrays={}\n        for model in MODELS:\n            with np.load(Path(root)/\'maps\'/archive_name(image,model,\'class\'),allow_pickle=False) as data:\n                arrays[model]={k:data[k] for k in (\'density_full\',\'fused_attention\',\'sim_x2\',\'sim_x1\')}\n        probabilities={m:normalize_density(a[\'density_full\'][0,0]) for m,a in arrays.items()}\n        pmax=max([float(p.max()) for p,status in probabilities.values() if p is not None] or [1.])\n        fig,axes=plt.subplots(3,3,figsize=(12,10),constrained_layout=True)\n        for i,model in enumerate(MODELS):\n            row=lookup.loc[(image,model,\'class\')]\n            density=arrays[model][\'density_full\'][0,0]\n            axes[i,0].imshow(rgb)\n            axes[i,0].set_title(\'{} | GT={:g}\\npred={:.2f}, abs error={:.2f}\'.format(model,row.gt_count,row.pred_count,row.abs_error),fontsize=10)\n            norm=PowerNorm(.5,vmin=0,vmax=raw_max) if raw_min>=0 else Normalize(vmin=raw_min,vmax=raw_max)\n            raw=axes[i,1].imshow(density,cmap=\'magma\' if raw_min>=0 else \'coolwarm\',norm=norm)\n            axes[i,1].set_title(\'Raw D, mass={:.2f}\\nshared all-20 scale; {}\'.format(density.sum(dtype=np.float64),\n                                \'sqrt display\' if raw_min>=0 else \'linear, negatives retained\'),fontsize=9)\n            p,status=probabilities[model]\n            if p is None:\n                axes[i,2].text(.5,.5,\'P unavailable:\\n\'+status,ha=\'center\',va=\'center\',transform=axes[i,2].transAxes)\n            else:\n                shape=axes[i,2].imshow(p,cmap=\'viridis\',vmin=0,vmax=pmax)\n                fig.colorbar(shape,ax=axes[i,2],label=\'Probability / pixel\',shrink=.65)\n            axes[i,2].set_title(\'P = D / sum(D)\\nshared across these three conditions\',fontsize=9)\n            fig.colorbar(raw,ax=axes[i,1],label=\'Count / pixel\',shrink=.65)\n            for ax in axes[i]: ax.axis(\'off\')\n        fig.suptitle(\'{} | target: {} | class-name inference\\nMagnitude and normalized shape are separate; no valid target dots\'.format(image,record[\'class\']))\n        save_figure(fig,output/\'figures\'/(\'qual_\'+Path(image).stem+\'_density.png\'),qa)\n        for index,(x,y,w,h) in enumerate(record[\'patch_xywh\']):\n            fig,axes=plt.subplots(3,4,figsize=(14,9),constrained_layout=True)\n            for i,model in enumerate(MODELS):\n                row=lookup.loc[(image,model,\'class\')]\n                for j,kind in enumerate((\'fused_attention\',\'sim_x2\',\'sim_x1\')):\n                    low,high,cmap=(0,1,\'viridis\') if j==0 else (-1,1,\'coolwarm\')\n                    rendered=axes[i,j].imshow(arrays[model][kind][index,0],vmin=low,vmax=high,cmap=cmap,\n                                              extent=(x,x+w,y+h,y))\n                    axes[i,j].set_title(model+\' / \'+kind,fontsize=9)\n                    fig.colorbar(rendered,ax=axes[i,j],shrink=.65)\n                    axes[i,j].set_xlabel(\'Original image x\',fontsize=8)\n                    axes[i,j].set_ylabel(\'Original image y\',fontsize=8)\n                axes[i,3].imshow(rgb[y:y+h,x:x+w])\n                axes[i,3].set_title(\'GT={:g}, full-image pred={:.2f}\\nabs error={:.2f}\'.format(row.gt_count,row.pred_count,row.abs_error),fontsize=9)\n                axes[i,3].axis(\'off\')\n            fig.suptitle(\'{} | {} | patch {} xywh={}\\nNative 48x48 intermediate maps placed in original patch coordinates; no reconstruction/localization claim\'.format(image,record[\'class\'],index,(x,y,w,h)))\n            save_figure(fig,output/\'figures\'/(\'qual_\'+Path(image).stem+\'_patch{:02d}.png\'.format(index)),qa)\n    return {\'raw_density_global_min\':raw_min,\'raw_density_global_max\':raw_max,\'raw_display_gamma\':.5 if raw_min>=0 else None,\n            \'raw_scale_scope\':\'all 20 class-prompt images x three models\',\n            \'normalized_shape_scale\':\'shared three conditions within each image, probability/pixel\'}\n\n\ndef markdown_table(frame,columns):\n    def display(value):\n        if value is None or pd.isna(value): return \'NA\'\n        if isinstance(value,(float,np.floating)): return \'{:.4f}\'.format(value)\n        return str(value).replace(\'|\',\'/\')\n    return \'\\n\'.join([\'| \'+\' | \'.join(columns)+\' |\',\'| \'+\' | \'.join([\'---\']*len(columns))+\' |\']+\n                     [\'| \'+\' | \'.join(display(row[column]) for column in columns)+\' |\'\n                      for row in frame.to_dict(\'records\')])\n\n\ndef research_report(output,tables,statuses,manifest,config):\n    counts=tables[\'counting_summary\'];overall=counts[counts.gt_bin==\'all\']\n    class_counts=overall[overall.prompt_type==\'class\'].set_index(\'model\')\n    pairs=tables[\'paired_counting_summary\'];class_pairs=pairs[(pairs.prompt_type==\'class\')&(pairs.gt_bin==\'all\')]\n    prompt=tables[\'prompt_sensitivity\']\n    prompt_summary=[]\n    for model,data in prompt.groupby(\'model\',sort=True):\n        prompt_summary.append({\'model\':model,\'n\':len(data),\'mean_delta_pred\':data.delta_pred.mean(),\n            \'median_delta_pred\':data.delta_pred.median(),\'mean_delta_abs_error\':data.delta_abs_error.mean(),\n            \'detailed_worse_n\':int((data.delta_abs_error>TIE_TOLERANCE).sum())})\n    prompt_summary=pd.DataFrame(prompt_summary)\n    shape=tables[\'density_shape_pairs\']\n    shape_class=shape[(shape.left_prompt==\'class\')&(shape.right_prompt==\'class\')]\n    shape_summary=[]\n    for (reference,candidate),data in shape_class.groupby([\'left_model\',\'right_model\']):\n        valid=data[data.shape_valid]\n        valid_r=valid.shape_pearson.dropna()\n        valid_ratio=data.mass_ratio.dropna()\n        shape_summary.append({\'reference\':reference,\'candidate\':candidate,\'n\':len(data),\'valid_shape_n\':len(valid),\n            \'median_mass_ratio\':valid_ratio.median() if len(valid_ratio) else None,\n            \'median_TV\':valid.total_variation.median() if len(valid) else None,\n            \'TV_min\':valid.total_variation.min() if len(valid) else None,\n            \'TV_max\':valid.total_variation.max() if len(valid) else None,\n            \'shape_pearson_valid_n\':len(valid_r),\n            \'median_shape_pearson\':valid_r.median() if len(valid_r) else None})\n    shape_summary=pd.DataFrame(shape_summary)\n    associations=tables[\'spatial_associations\']\n    key_associations=associations[(associations.kind==\'checkpoint\')&\n        associations.comparison.str.contains(\'/class -> \',regex=False)&associations.comparison.str.endswith(\'/class\')&\n        (associations.feature==\'normalized_l1\')&(associations.gt_bin==\'all\')&\n        (associations.method==\'spearman\')&(associations.adjustment==\'log1p_gt_and_reference_error\')]\n    base_to_nine=class_pairs[(class_pairs.reference==\'BASE-002\')&(class_pairs.candidate==\'RP-009\')].iloc[0]\n    invalid=sum(row[\'normalization_status\']!=\'valid\' for row in statuses)\n    gt_bins=counts[(counts.model==\'BASE-002\')&(counts.prompt_type==\'class\')&(counts.gt_bin!=\'all\')][[\'gt_bin\',\'n\']]\n    sections=[\n        \'# DIAG-002 - CPU post-hoc research report (descriptive; researcher review required)\',\n        \'\\nThis report uses already-generated DIAG-001 artifacts only. No model/checkpoint was loaded, no inference or training was run. Class-name inference is primary. Detailed prompting is a diagnostic perturbation. Dot localization accuracy is unavailable.\',\n        \'\\n## Coverage and provenance\',\n        \'\\n230 images; 1,380 exact image/model/prompt records; 14,490 keyed spatial comparisons; 120 native-map NPZs on the original 20. Input hashes and source-code hash are in analysis_manifest.json. Source DIAG-001 commit: `\'+str(manifest.get(\'source\',{}).get(\'commit\',\'unrecorded\'))+\'`.\',\n        \'\\nGT bin sizes (the high-count group is small; inspect n rather than interpreting its mean as stable):\\n\',\n        markdown_table(gt_bins,[\'gt_bin\',\'n\']),\n        \'\\n## Finding A - Counting calibration\',\n        \'\\n**Direct evidence.** Full-230 class-name counting summaries:\',\n        markdown_table(class_counts.reset_index(),[\'model\',\'n\',\'mae\',\'rmse\',\'signed_bias\',\'median_absolute_error\',\'over_n\',\'over_pct\',\'under_n\',\'under_pct\',\'pred_mean\']),\n        \'\\nPaired candidate wins/losses and conditional image-bootstrap mean error-gain intervals:\',\n        markdown_table(class_pairs,[\'reference\',\'candidate\',\'n\',\'candidate_wins\',\'candidate_loses\',\'ties\',\'mean_error_gain\',\'gain_ci_low\',\'gain_ci_high\',\'mean_delta_pred\',\'prediction_lower_n\',\'mean_over_error_gain\',\'mean_under_error_gain\']),\n        \'\\n**Arithmetic decomposition, not causal attribution.** BASE-002 -> RP-009: mean error gain = {:.4f}; reduction of positive overcount error = {:.4f}; reduction of undercount error = {:.4f}. The two signed contributions sum to the gain. RP-009 lowers predictions on {}/230 images.\'.format(\n            base_to_nine.mean_error_gain,base_to_nine.mean_over_error_gain,base_to_nine.mean_under_error_gain,int(base_to_nine.prediction_lower_n)),\n        \'\\n**Association.** prediction_distributions.png, paired_counting_gains.png and gt_bin_magnitude_bias.png separate magnitude, signed error and GT strata. Full distributions/per-image values remain in the tables.\',\n        \'\\n**Plausible interpretation.** Reduced overcount burden is consistent with a calibration/magnitude component. It does not demonstrate improved object recognition. The density-shape evidence below tests whether a pure multiplicative description is sufficient on the saved subset.\',\n        \'\\n**Alternatives and uncertainty.** Training/procedure/embedding changes, scene composition and the small high-count bin can affect both predictions and error. Bootstrap intervals resample images with seed {} ({} repetitions); they condition on this dataset and these single trained checkpoints, not checkpoint/seed uncertainty or unrelated datasets.\'.format(config[\'seed\'],config[\'bootstrap_repetitions\']),\n        \'\\n## Finding B - Spatial redistribution\',\n        \'\\n**Direct evidence (original 20 only).** D is the saved evaluator-reassembled density. P=D/sum(D) is calculated only for finite nonnegative D with mass > {}. No clipping, smoothing or pseudo annotations. Invalid normalization conditions among all 120 archives: {}.\'.format(EPSILON,invalid),\n        markdown_table(shape_summary,[\'reference\',\'candidate\',\'n\',\'valid_shape_n\',\'median_mass_ratio\',\'median_TV\',\'TV_min\',\'TV_max\',\'median_shape_pearson\']),\n        \'\\n**Association.** density_mass_vs_shape.png plots mass ratios separately from total variation. TV=0 denotes identical normalized distributions; differences above floating-point tolerance cannot be produced by scalar multiplication alone.\',\n        \'\\n**Plausible interpretation.** Similar normalized shape with lower mass is consistent with magnitude change; changed normalized shape indicates redistribution. Neither establishes that mass moved toward the requested target.\',\n        \'\\n**Alternatives and uncertainty.** Shape changes may suppress or amplify distractors. The 20 images were selected before inference with count/class coverage and are not frequency-representative of all 230. Raw maps are unavailable for the other 210, so no raw-shape reconstruction or extrapolated localization claim is made.\',\n        \'\\n## Finding C - U-Net versus HSCM\',\n        \'\\n**Direct evidence / statistical association.** Partial Spearman between normalized map L1 and class-name counting gain, adjusting ranked variables for log1p(GT) and log1p(reference absolute error):\',\n        markdown_table(key_associations,[\'comparison\',\'map_type\',\'n\',\'r\',\'status\']),\n        \'\\nRaw Pearson/Spearman, 1-minus-spatial-Pearson features, GT strata and valid counts are in spatial_associations.csv. Better/worse/tie distributions are in spatial_groups.csv. Undefined spatial Pearson values are omitted only from calculations requiring them and their valid n is reported; normalized L1 still uses all valid rows.\',\n        \'\\n**Plausible interpretation.** Consistently stronger HSCM associations across pairs/adjustment would be compatible with an HSCM-related hypothesis. Compare both similarity stages against every attention resolution; the pipeline does not select the largest coefficient as a proven mechanism.\',\n        \'\\n**Alternatives and uncertainty.** GT and reference error can confound associations; reference error also appears algebraically in error_gain, so residualization does not create causal identification. Same-map minmax L1 is not raw magnitude. Attention/HSCM/density are not interchangeable physical scores. Associations are exploratory, correlated and numerous; no p-value screening or causal model ranking is performed.\',\n        \'\\n## Finding D - Prompt sensitivity\',\n        \'\\n**Direct evidence.** Within-checkpoint detailed minus class changes:\',\n        markdown_table(prompt_summary,[\'model\',\'n\',\'mean_delta_pred\',\'median_delta_pred\',\'mean_delta_abs_error\',\'detailed_worse_n\']),\n        \'\\n**Statistical association.** prompt_association_matrix.png relates within-model count change to map changes, raw and adjusted. spatial_associations.csv separately includes delta_abs_error as a target. These joins compare each image with itself under its exact saved prompt; no cross-checkpoint result is substituted for a prompt effect.\',\n        \'\\n**Plausible interpretation.** Count increases accompanied by attention/HSCM/density changes are compatible with prompt-sensitive behavior. Descriptions change content, token count and embeddings together; no claim isolates text length or identifies why a particular internal stage causes failure.\',\n        \'\\n**Alternatives and uncertainty.** Token-mean attention for different sequences has different semantic content. Dataset-specific targets, prompt composition and training provenance remain alternative explanations. The FSC147-S ranking reversal does not establish that generic templates are generally superior; standard FSC147 test evidence differs.\',\n        \'\\n## Researcher decision table (no automatic selection)\',\n        \'\\n| Future direction | Evidence relevant to review | Still needed before a scientific decision |\\n| --- | --- | --- |\\n| Density calibration / counting objective | Bias, overcount decomposition, mass-versus-shape plots | A separately approved controlled intervention and held-out counting evaluation |\\n| HSCM semantic fusion | Both-stage associations before/after adjustment | Provenance-matched causal control; no conclusion from correlation alone |\\n| U-Net conditioning / cross-attention | Resolution-specific and prompt-paired associations | Token-semantic controls and a separately approved intervention |\\n| No architectural change yet | Single checkpoints, confounding, missing target spatial GT | Researcher review, replication/provenance and additional evidence |\',\n        \'\\nThe final decision belongs to the researcher. PROJECT_STATE.md and EXPERIMENT_LEDGER.md are not edited by this pipeline.\'\n    ]\n    (output/\'research_report.md\').write_text(\'\\n\'.join(sections)+\'\\n\',encoding=\'utf-8\')\n    return {\'class_counting\':class_counts.reset_index().to_dict(\'records\'),\n            \'class_paired\':class_pairs.to_dict(\'records\'),\'prompt_sensitivity\':prompt_summary.to_dict(\'records\'),\n            \'density_shape_class20\':shape_summary.to_dict(\'records\'),\'normalization_invalid_n\':invalid,\n            \'gt_bins\':gt_bins.to_dict(\'records\')}\n\n\ndef input_inventory(root):\n    root=Path(root)\n    paths=[root/name for name in (\'manifest.json\',\'summary.json\',\'results.csv\',\'map_comparisons.csv\')]\n    paths+=sorted((root/\'maps\').glob(\'*.npz\'))\n    return {p.relative_to(root).as_posix():{\'sha256\':sha256(p),\'size_bytes\':p.stat().st_size} for p in paths}\n\n\ndef _code_hash():\n    return hashlib.sha256(Path(__file__).read_bytes().replace(b\'\\r\\n\',b\'\\n\')).hexdigest()\n\n\ndef _source_commit():\n    try:\n        return subprocess.check_output([\'git\',\'rev-parse\',\'HEAD\'],stderr=subprocess.DEVNULL,text=True).strip()\n    except (OSError,subprocess.CalledProcessError):\n        return None\n\n\ndef run_analysis(input_dir,output_dir=None,image_root=None,bootstrap_repetitions=2000,seed=3407,\n                 reuse=False,validate_only=False):\n    if seed!=3407:\n        raise ValueError(\'Protocol seed is fixed at 3407\')\n    if bootstrap_repetitions<1:\n        raise ValueError(\'bootstrap_repetitions must be positive\')\n    root=Path(input_dir).resolve()\n    output=Path(output_dir).resolve() if output_dir else root/\'analysis_cpu_v1\'\n    if root==output or output in root.parents or root/\'maps\'==output or root/\'figures\'==output:\n        raise ValueError(\'Analysis output must not replace an input directory\')\n    log=logging.getLogger(\'diag002_cpu\');log.setLevel(logging.INFO)\n    console=logging.StreamHandler(sys.stdout);log.addHandler(console)\n    file_handler=None\n    state=None\n    config={\'seed\':seed,\'bootstrap_repetitions\':bootstrap_repetitions,\n            \'density_mass_epsilon\':EPSILON,\'tie_tolerance\':TIE_TOLERANCE,\n            \'confounders\':[\'log1p(GT)\',\'log1p(reference absolute error)\']}\n    try:\n        log.info(\'DIAG-002: reading immutable saved artifacts (CPU only, no model imports)\')\n        before=input_inventory(root)\n        manifest,source_summary,results,maps=validate_tables(root)\n        log.info(\'Validated keyed table coverage: 230 images / 1380 predictions / 14490 map pairs\')\n        statuses,images=validate_npz(root,manifest,results,image_root)\n        image_fingerprints={image:{\'path\':str(path),\'sha256\':sha256(path)} for image,path in images.items()}\n        log.info(\'Validated 120 native NPZ schemas, density mass/scaling and 20 matching RGB images\')\n        if validate_only:\n            return {\'status\':\'validated\',\'images\':230,\'conditions\':1380,\'map_comparisons\':14490,\n                    \'archives\':120,\'normalization_statuses\':statuses}\n        if output.exists():\n            if not reuse:\n                raise FileExistsError(\'Analysis output exists; choose a new name or explicitly use --reuse: \'+str(output))\n            try:\n                saved=json.loads((output/\'analysis_manifest.json\').read_text(encoding=\'utf-8\'))\n                if saved[\'status\']!=\'complete\' or saved[\'input_files\']!=before or saved[\'config\']!=config or saved[\'code_sha256\']!=_code_hash():\n                    raise ValueError(\'Existing report is incomplete or input/config/source hash differs\')\n                for relative,info in saved[\'output_files\'].items():\n                    if sha256(output/relative)!=info[\'sha256\']:\n                        raise ValueError(\'Existing output changed: \'+relative)\n                if input_inventory(root)!=before:\n                    raise ValueError(\'Inputs changed during reuse validation\')\n            except (OSError,ValueError,KeyError) as exc:\n                raise ValueError(\'Cannot reuse safely; preserve this directory and choose a new output: \'+str(exc)) from exc\n            log.info(\'Reused complete verified analysis; no files overwritten: %s\',output)\n            return json.loads((output/\'summary.json\').read_text(encoding=\'utf-8\'))\n        output.mkdir(parents=True,exist_ok=False)\n        (output/\'figures\').mkdir()\n        file_handler=logging.FileHandler(output/\'run.log\',encoding=\'utf-8\')\n        log.addHandler(file_handler)\n        state={\'schema_version\':SCHEMA,\'status\':\'running\',\'created_at\':datetime.now(timezone.utc).isoformat(),\n               \'input_dir\':str(root),\'output_dir\':str(output),\'config\':config,\'input_files\':before,\n               \'images\':image_fingerprints,\'code_sha256\':_code_hash(),\'source_commit\':_source_commit(),\n               \'source_commit_note\':\'Base checkout only; analysis implementation is identified by its own SHA256\',\n               \'execution\':{\'device\':\'CPU\',\'model_loading\':False,\'checkpoint_loading\':False,\'additional_inference\':False},\n               \'environment\':{\'python\':sys.version,\'packages\':{name:importlib.metadata.version(name)\n                    for name in (\'numpy\',\'pandas\',\'scipy\',\'matplotlib\',\'Pillow\')}}}\n        json_write(output/\'analysis_manifest.json\',state)\n        json_write(output/\'input_validation.json\',{\'status\':\'passed\',\'images\':230,\'conditions\':len(results),\n                    \'map_comparisons\':len(maps),\'archives\':120,\'normalization\':statuses,\n                    \'join_keys\':[\'image\',\'exact model\',\'exact prompt\'],\'input_repair\':False})\n        log.info(\'Computing paired counting, conditional bootstrap and spatial associations\')\n        tables=counting_tables(results,bootstrap_repetitions)\n        tables.update(spatial_tables(results,maps))\n        tables[\'density_normalization\']=pd.DataFrame(statuses)\n        tables[\'density_shape_pairs\']=density_shapes(root,statuses)\n        for name,data in tables.items(): csv_write(output/(name+\'.csv\'),data)\n        # Prompt length is not manipulated or interpreted causally.\n        qa=[]\n        import matplotlib\n        matplotlib.use(\'Agg\',force=True)\n        aggregate_plots(output,results,tables,qa)\n        log.info(\'Rendering original-20 qualitative panels with native patch coordinates\')\n        display=qualitative_plots(root,output,manifest,results,images,qa)\n        json_write(output/\'figure_qa.json\',qa)\n        summary=research_report(output,tables,statuses,manifest,config)\n        summary.update({\'schema_version\':SCHEMA,\'status\':\'complete\',\'evaluation_images\':230,\n                        \'raw_map_images\':20,\'tables\':{k:len(v) for k,v in tables.items()},\'figure_count\':len(qa),\n                        \'display\':display,\'interpretation\':\'Descriptive associations; no causal or localization-accuracy conclusion. Researcher decision required.\'})\n        json_write(output/\'summary.json\',summary)\n        if input_inventory(root)!=before or any(sha256(p[\'path\'])!=p[\'sha256\'] for p in image_fingerprints.values()):\n            raise RuntimeError(\'Input artifact changed during analysis; outputs not marked complete\')\n        state[\'input_immutability_verified\']=True\n        log.info(\'Completed CPU analysis; all %d figures rendered/checked; inputs unchanged\',len(qa))\n        log.removeHandler(file_handler);file_handler.close();file_handler=None\n        state[\'output_files\']={p.relative_to(output).as_posix():{\'sha256\':sha256(p),\'size_bytes\':p.stat().st_size}\n                               for p in sorted(output.rglob(\'*\')) if p.is_file() and p.name!=\'analysis_manifest.json\'}\n        state[\'status\']=\'complete\'\n        json_write(output/\'analysis_manifest.json\',state)\n        return summary\n    except BaseException:\n        if state is not None:\n            state[\'status\']=\'failed\';json_write(output/\'analysis_manifest.json\',state)\n        log.exception(\'DIAG-002 failed; inputs are not repaired and partial outputs remain for inspection\')\n        raise\n    finally:\n        if file_handler is not None:\n            log.removeHandler(file_handler);file_handler.close()\n        log.removeHandler(console);console.close()\n\n\ndef parse_args(argv=None):\n    parser=argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--input-dir\',required=True)\n    parser.add_argument(\'--output-dir\')\n    parser.add_argument(\'--image-root\',help=\'Existing original RGB directory; every image must match manifest SHA256\')\n    parser.add_argument(\'--bootstrap-repetitions\',type=int,default=2000)\n    parser.add_argument(\'--seed\',type=int,default=3407)\n    parser.add_argument(\'--reuse\',action=\'store_true\',help=\'Read a completed matching output after validating every output hash\')\n    parser.add_argument(\'--validate-only\',action=\'store_true\')\n    return parser.parse_args(argv)\n\n\ndef main(argv=None):\n    args=parse_args(argv)\n    summary=run_analysis(args.input_dir,args.output_dir,args.image_root,args.bootstrap_repetitions,\n                         args.seed,args.reuse,args.validate_only)\n    print(json.dumps({\'status\':summary[\'status\'],\'output\':str(Path(args.output_dir) if args.output_dir else Path(args.input_dir)/\'analysis_cpu_v1\')},indent=2))\n\n\nif __name__==\'__main__\':\n    main()\n'

assert hashlib.sha256(EMBEDDED_ANALYSIS_SOURCE.replace('\r\n','\n').encode()).hexdigest() == EXPECTED_ANALYSIS_SHA256
if LOCAL_ANALYSIS_SCRIPT.is_file():
    assert hashlib.sha256(LOCAL_ANALYSIS_SCRIPT.read_bytes().replace(b'\r\n',b'\n')).hexdigest() == EXPECTED_ANALYSIS_SHA256, 'Local helper differs from this reviewed notebook; synchronize the two sources before analysis.'
    script_path=LOCAL_ANALYSIS_SCRIPT
else:
    _cpu_source_dir=tempfile.TemporaryDirectory(prefix='diag002_cpu_source_')
    atexit.register(_cpu_source_dir.cleanup)
    script_path=Path(_cpu_source_dir.name)/'analyze_spatial_diagnostic.py'
    script_path.write_text(EMBEDDED_ANALYSIS_SOURCE,encoding='utf-8')
spec=importlib.util.spec_from_file_location('diag002_cpu_analysis',script_path)
analysis=importlib.util.module_from_spec(spec)
spec.loader.exec_module(analysis)
print('Reviewed analysis source SHA256:',EXPECTED_ANALYSIS_SHA256)


## Validate exact table joins and locate the original 20 RGB images
The RGB files are context for qualitative panels, not extra inference or annotations.
Every RGB must match its DIAG-001 manifest hash. If the original runtime images are
gone, the cell can read just those 20 existing JPEGs from the saved asset ZIP; it never
extracts model weights. Configure IMAGE_ROOT explicitly if needed.


In [ ]:
source_manifest, source_summary, predictions, map_metrics = analysis.validate_tables(INPUT_DIR)
print('Table coverage:', predictions.image.nunique(), 'images;', len(predictions), 'predictions;', len(map_metrics), 'map rows')
records={record['image']:record for record in source_manifest['selected_images']}
if IMAGE_ROOT is None and not all(Path(records[i]['image_file']['path']).is_file() for i in analysis.ORIGINAL_IMAGES):
    candidates=[ASSET_ROOT/'datasets/FSC147/images_384_VarV2',
                Path('/content/T2ICount-assets/datasets/FSC147/images_384_VarV2')]
    if os.environ.get('T2ICOUNT_ASSET_ROOT'):
        candidates.append(Path(os.environ['T2ICOUNT_ASSET_ROOT'])/'datasets/FSC147/images_384_VarV2')
    IMAGE_ROOT=next((p for p in candidates if all((p/i).is_file() for i in analysis.ORIGINAL_IMAGES)),None)
    if IMAGE_ROOT is None:
        assert ASSET_ZIP.is_file(), 'Set IMAGE_ROOT to the matching existing 20 RGB images, or provide the existing asset ZIP.'
        _cpu_image_dir=tempfile.TemporaryDirectory(prefix='diag002_cpu_rgb_')
        atexit.register(_cpu_image_dir.cleanup)
        IMAGE_ROOT=Path(_cpu_image_dir.name)
        with zipfile.ZipFile(ASSET_ZIP) as archive:
            names=archive.namelist()
            for image in analysis.ORIGINAL_IMAGES:
                matches=[name for name in names if 'images_384_VarV2/' in name and Path(name).name==image]
                assert len(matches)==1, f'Ambiguous/missing original JPEG in asset ZIP: {image}'
                with archive.open(matches[0]) as src, (IMAGE_ROOT/image).open('xb') as dst:
                    shutil.copyfileobj(src,dst)
normalization_statuses, image_paths=analysis.validate_npz(INPUT_DIR,source_manifest,predictions,IMAGE_ROOT)
print('Validated 120 native NPZs, density sums/scaling, token means, patch coordinates and 20 RGB hashes.')
print('Normalization status counts:', __import__('collections').Counter(r['normalization_status'] for r in normalization_statuses))


## Run or safely reuse CPU analysis
The default output is a new `analysis_cpu_v1` child of the existing DIAG-001 folder.
Existing complete matching outputs are read after checking input/config/code/output
hashes. A partial, modified, or incompatible analysis is preserved and refused: choose
a new OUTPUT_DIR. No resume, deletion, or implicit overwrite is performed.


In [ ]:
if RUN_ANALYSIS:
    summary=analysis.run_analysis(INPUT_DIR,OUTPUT_DIR,IMAGE_ROOT,
        bootstrap_repetitions=BOOTSTRAP_REPETITIONS,seed=SEED,reuse=REUSE_COMPLETED,validate_only=VALIDATE_ONLY)
    print('Status:',summary['status'])
else:
    print('CPU analysis disabled. Existing saved reports can be displayed below.')


## Counting and paired comparisons
GT-bin sample sizes are reported explicitly. Bootstrap intervals are image-conditional,
not training-seed uncertainty or proof of causal/generalizable effects.


In [ ]:
from IPython.display import display, Image as DisplayImage, Markdown
import pandas as pd
if (OUTPUT_DIR/'summary.json').is_file():
    summary=json.loads((OUTPUT_DIR/'summary.json').read_text(encoding='utf-8'))
    display(pd.DataFrame(summary['class_counting']))
    display(pd.DataFrame(summary['class_paired']))
    display(pd.DataFrame(summary['gt_bins']))
    for name in ('count_vs_gt.png','paired_counting_gains.png','gt_bin_magnitude_bias.png'):
        display(DisplayImage(filename=str(OUTPUT_DIR/'figures'/name)))
else:
    print('No completed analysis report yet.')


## Spatial associations and density shape
Raw-map analysis uses only the original 20, whose maps exist. The other 210 contribute
saved comparison metrics only. Partial Spearman residualizes ranked variables against
ranked log1p(GT) and log1p(reference error); this does not eliminate all confounding or
the algebraic coupling of reference error with error_gain.


In [ ]:
if (OUTPUT_DIR/'summary.json').is_file():
    display(pd.DataFrame(summary['density_shape_class20']))
    for name in ('spatial_association_matrix.png','prompt_association_matrix.png','density_mass_vs_shape.png'):
        display(DisplayImage(filename=str(OUTPUT_DIR/'figures'/name)))
    VIEW_IMAGE=analysis.ORIGINAL_IMAGES[0]  # Pick only an ID in the original fixed set.
    assert VIEW_IMAGE in analysis.ORIGINAL_IMAGES
    for path in sorted((OUTPUT_DIR/'figures').glob('qual_'+Path(VIEW_IMAGE).stem+'_*.png')):
        display(DisplayImage(filename=str(path)))


## Research report and validation
Read the four findings as observed evidence, associations, plausible interpretations,
alternatives, and uncertainty. The final research decision belongs to the researcher;
this notebook never updates PROJECT_STATE.md or EXPERIMENT_LEDGER.md.


In [ ]:
if (OUTPUT_DIR/'research_report.md').is_file():
    display(Markdown((OUTPUT_DIR/'research_report.md').read_text(encoding='utf-8')))
    completed=json.loads((OUTPUT_DIR/'analysis_manifest.json').read_text(encoding='utf-8'))
    print('Input immutability verified:', completed['input_immutability_verified'])
    print('Execution:',completed['execution'])
    print('All figures rendered and checked:',summary['figure_count'])
    print('Saved output:',OUTPUT_DIR)
